# Laboratório de evidências — segurança rodoviária e governança
**Edição de 17/10/2026.** Reprodução das extensões analíticas elaboradas para avaliação de uma proposta de tese. Não representa resultados produzidos pela autora da proposta nem identificação causal de governança.

Execute em ordem em **Ambiente de execução → Executar tudo**. Os arquivos originais são obtidos de uma versão preservada no GitHub; o manifesto informa a origem oficial e SHA-256. O processamento completo requer alguns minutos e conexão para baixar aproximadamente 115 MB.

## Mapa da análise
1. Ambiente e fontes preservadas; 2. Auditoria PRF; 3. SNV e traçado; 4. Exposição PNCT; 5. modelos e sensibilidades; 6. validação; 7. governança e limitações.

O notebook mostra o **código integral executado** em cada etapa. Comentários e interpretação acompanham as fórmulas. Alterar o código ou a versão das fontes produz outra análise e deve ser documentado.

In [1]:
# Instalação somente quando necessário. Os limites das versões estão no repositório.
import sys, subprocess, importlib.util, pathlib, os
if importlib.util.find_spec('google.colab') is not None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests>=2.32,<3', 'pandas>=2.2,<4', 'numpy>=2,<3', 'scipy>=1.14,<2', 'matplotlib>=3.9,<4'], check=True)
ROOT = pathlib.Path.cwd()
if not (ROOT/'scripts/analisar.py').exists():
    ROOT = pathlib.Path('/content/observatorio-seguranca-rodoviaria')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', 'https://github.com/alexandrerodrigues2311/observatorio-seguranca-rodoviaria.git', str(ROOT)], check=True)
os.chdir(ROOT)
print('Diretório de reprodução:', ROOT)
print(subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True).stdout.strip() or 'Cópia local em preparação')


Diretório de reprodução: /projeto/observatorio-seguranca-rodoviaria
HEAD


## 1. Originais e rastreabilidade
O download abaixo preserva a fotografia usada na análise. Não substitui automaticamente as fontes por versões atuais. Cada arquivo tem origem oficial no manifesto. O material da proposta de tese e quaisquer credenciais não fazem parte do pacote.

Arquivos PRF: ocorrências por ano; SNV: cadastro e geometrias; PNCT: planilhas VMD e notas técnicas; ANTT: equipamentos, investimentos e tráfego em pedágios. As passagens em pedágios **não** são usadas como exposição por trecho.

In [2]:
import requests, zipfile, hashlib, json
if not (ROOT/'data/manifesto.json').exists():
    url = 'https://github.com/alexandrerodrigues2311/observatorio-seguranca-rodoviaria/releases/download/dados-2026-10-17/dados-oficiais-2026-10-17.zip'
    response = requests.get(url, timeout=180); response.raise_for_status()
    import io
    with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
        # Extração limitada à pasta data; impede caminhos que escapem do projeto.
        for name in archive.namelist():
            target = (ROOT/name).resolve()
            if not target.is_relative_to((ROOT/'data').resolve()):
                raise ValueError('Caminho inesperado no pacote: '+name)
        archive.extractall(ROOT)
manifest = json.loads((ROOT/'data/manifesto.json').read_text(encoding='utf-8'))
for source in manifest:
    path = ROOT/'data'/source['file']
    if source.get('sha256'):
        assert path.exists(), source['file']
        assert hashlib.sha256(path.read_bytes()).hexdigest()==source['sha256'], source['file']
print('Integridade SHA-256 verificada:', len(manifest), 'fontes do manifesto')
import pandas as pd, numpy as np
print(pd.DataFrame(manifest)[['file','bytes','url']].to_string(index=False))


Integridade SHA-256 verificada: 26 fontes do manifesto
                                    file    bytes                                                                                                                                                                                        url
         antt-acompanhamento-anual-0.csv   426435                       https://dados.antt.gov.br/dataset/b4ef619c-71a1-48a5-86c8-1109964753c5/resource/c6fe24da-3fdf-4a3b-8d96-c5d30c90290d/download/acompanhamento-anual-investimentos.csv
      antt-cadastros-investimentos-0.csv  4690578    https://dados.antt.gov.br/dataset/6a42582f-5481-458d-bf21-247d91a60af5/resource/48ca5a9b-65c2-46af-9123-cad23e6f8a8d/download/cadastros-de-investimentos-dos-contratos-de-concessao.csv
                      antt-catalogo.json   817164                                                                                                         https://dados.antt.gov.br/api/3/action/package_search?fq=groups:rodovias&rows=10

## 2. Auditoria e análise das ocorrências PRF
**Unidade:** uma ocorrência registrada, não uma pessoa. Os identificadores são verificados dentro de cada ano. Campos numéricos definem mortes e feridos; divergências da classificação textual e da soma de pessoas são auditadas, sem imputação automática.

**Gravidade:** Y=1 quando mortos>0. A regressão logística estima `logit P(Y=1 | registro, X) = Xβ`; `exp(β)` é uma razão de chances **entre ocorrências registradas**, não risco de sofrer sinistro. Controles: noite, urbano, pista dupla/múltipla, chuva, fim de semana, UF e ano. A matriz sanduíche agrupa os escores por UF–BR, com correção finita e quantil t.

**Persistência:** intervalos UF–BR de 10 km; Spearman entre mortes de dois biênios, IC por 500 reamostragens de corredores, semente 20261017. O universo seleciona intervalos com registro em algum ano, inclusive posterior: não é validação prospectiva.

In [3]:
# O mesmo arquivo executado pelo painel é mostrado integralmente, sem pseudocódigo.
# No Colab, Code permite leitura com destaque. A execução usa um processo isolado.
try:
    from IPython.display import display, Code
except ImportError:
    display = print
    Code = lambda text, **kwargs: text

def executar_etapa(nome):
    path = ROOT/'scripts'/nome
    display(Code(path.read_text(encoding='utf-8-sig'), language='python'))
    result = subprocess.run([sys.executable, '-u', str(path)], cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError('Etapa interrompida: '+nome)

executar_etapa('analisar.py')


"""Análises de registros PRF, auditoria e inferência condicional reproduzíveis.
Não estima sub-registro nem efeitos causais. Sem dependência de statsmodels.
"""
from pathlib import Path
import json, zipfile, io, datetime, hashlib, unicodedata, re, math, csv
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.optimize import minimize
from scipy.stats import rankdata, t as student_t

ROOT=Path(__file__).resolve().parents[1]; RAW=ROOT/'data'; OUT=ROOT/'dist'/'data'
OUT.mkdir(parents=True,exist_ok=True); (ROOT/'dist'/'exports').mkdir(exist_ok=True)
def write(name,obj):
    text=json.dumps(obj,ensure_ascii=False,separators=(',',':'),allow_nan=False,default=lambda x:x.item() if hasattr(x,'item') else str(x))
    (OUT/(name+'.json')).write_text(text,encoding='utf8')
    (OUT/(name+'.js')).write_text('window.'+name.upper()+'='+text+';',encoding='utf8')
def norm(x):return ''.join(c for c in unicodedata.normalize('NFKD',str(x)) if not unicodedata.combining(c)).lower

In [4]:
def resultado(nome):
    return json.loads((ROOT/'dist/data'/f'{nome}.json').read_text(encoding='utf-8'))
print(pd.DataFrame(resultado('resumo')).to_string(index=False))
print(pd.DataFrame(resultado('auditoria')).to_string(index=False))
print('INTERPRETAÇÃO: os totais descrevem registros da versão preservada; não quantificam todos os eventos reais nem óbitos após acompanhamento de 30 dias.')


 ano  ocorrencias  mortos  feridos_graves  feridos_leves  fatais
2022        64606    5441           18173          54892    4664
2023        67766    5627           18958          59505    4858
2024        73202    6163           20372          64223    5225
2025        72529    6043           20018          63532    5210
 ano  linhas_brutas  ocorrencias  duplicados  ids_conflitantes  mortos  feridos_graves  feridos_leves  sinistros_fatais  sem_vitimas  classificacao_ausente  classificacao_divergente  localizacao_invalida  coordenada_invalida  data_invalida  ano_divergente  contagem_vitimas_divergente
2022          64606        64606           0                 0    5441           18173          54892              4664        11506                      1                         0                   199                    0              0               0                         3547
2023          67766        67766           0                 0    5627           18958          59505    

## 3. Vinculação SNV e traçados
Vinculação por UF, BR e km em `[início, fim)`, com inclusão do ponto final do corredor. Correspondências ambíguas são excluídas. O cadastro de 2026 serve para auditoria e visualização retrospectiva; não reconstrói mudanças históricas de administração.

Trecho sem registro é **zero observado na vinculação**, não prova de segurança ou sub-registro. O traçado simplificado é exclusivamente cartográfico.

In [5]:
executar_etapa('integrar_snv.py')
executar_etapa('geometria_rodovias.py')
print('Correspondências:', resultado('snv')['match'])

"""Diagnóstico espacial com SNV 202604A. Não reconstrói governança histórica."""
from pathlib import Path
import sys,zipfile,json
import pandas as pd
import numpy as np
ROOT=Path(__file__).resolve().parents[1];sys.path.insert(0,str(ROOT/'scripts'/'vendor'))
import shapefile
RAW=ROOT/'data';OUT=ROOT/'dist'/'data'
def write(name,obj):
 t=json.dumps(obj,ensure_ascii=False,separators=(',',':'),allow_nan=False,default=lambda x:x.item() if hasattr(x,'item') else str(x));(OUT/(name+'.json')).write_text(t,encoding='utf8');(OUT/(name+'.js')).write_text('window.'+name.upper()+'='+t+';',encoding='utf8')
z=zipfile.ZipFile(RAW/'snv202604A.zip');s=next(n for n in z.namelist() if n.endswith('.shp'))
r=shapefile.Reader(shp=z.open(s),shx=z.open(s[:-4]+'.shx'),dbf=z.open(s[:-4]+'.dbf'),encoding='utf8')
S=pd.DataFrame([x.as_dict() for x in r.iterRecords()])
print({c:S[c].value_counts().to_dict() for c in ['sg_tipo_tr','ds_jurisdi','ds_sup_fed','ds_tipo_ad']},flush=True)
S['br']=pd.to_numeric(S.vl_br,erro

## 4. Extração PNCT e denominador de exposição
A extração OOXML é somente leitura e preserva os arquivos originais. Usam-se os limites dos **segmentos estimados**, evitando repetir a extensão de um segmento original subdividido.

`VMD = VMDa_C + VMDa_D`; `Extensão = km_f_estim − km_i_estim`; `Exposição = VMD × Extensão × dias_do_ano`; `Taxa = desfecho / exposição × 100.000.000`.

**Elegibilidade:** eixo principal B, jurisdição federal, pavimentado ou duplicado, extensão e VMD positivos. O painel inclui os zeros de registro neste universo. A versão anual do cadastro PNCT é distinta do SNV cartográfico de 2026.

**Limitação relevante:** o VMD de 2025 usa julho/2024–junho/2025 conforme nota técnica. Modelos não propagam a incerteza da estimativa de tráfego. Há sensibilidade excluindo 2025.

In [6]:
executar_etapa('extrair_planilhas.py')

"""Extração somente leitura de valores OOXML, sem alterar as fontes."""
from pathlib import Path
import zipfile,xml.etree.ElementTree as ET,csv
R=Path(__file__).resolve().parents[1]
ns={'s':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
for year in range(2022,2026):
 p=R/'data'/f'pnct-vmda-{year}.xlsx';z=zipfile.ZipFile(p)
 shared=[]
 if 'xl/sharedStrings.xml' in z.namelist():
  root=ET.fromstring(z.read('xl/sharedStrings.xml'));shared=[''.join(e.itertext()) for e in root.findall('s:si',ns)]
 for name in [n for n in z.namelist() if n.startswith('xl/worksheets/sheet') and n.endswith('.xml')]:
  root=ET.fromstring(z.read(name));rows=[]
  for row in root.findall('s:sheetData/s:row',ns):
   values=[]
   for c in row.findall('s:c',ns):
    ref=c.attrib.get('r','A1');letters=''.join(x for x in ref if x.isalpha());col=0
    for letter in letters:col=col*26+ord(letter)-64
    while len(values)<col:values.append('')
    v=c.find('s:v',ns);value=v.text if v is not None else ''
    

## 5. Modelos binomiais negativos NB2
**Desfechos separados:** contagem de sinistros e contagem de mortes por segmento-ano. `E[Y]=μ`, `Var[Y]=μ+αμ²`. O modelo estima β e log α por máxima verossimilhança com `log μ = log(exposição/10⁸) + Xβ`.

O **offset** fixa o denominador; log VMD entra também como covariável para permitir elasticidade adicional. Controles: regime cadastral, pista, proporção de pesados em incrementos de 10 pontos percentuais, indicador não simulável, UF e ano. `IRR=exp(β)`. A Hessiana e escores individuais entram na covariância sanduíche por UF–BR. Covariáveis constantes são retiradas.

A dispersão de Pearson do Poisson é um diagnóstico; AIC compara ajustes nesta amostra, não demonstra causalidade. Os ICs condicionam o tráfego estimado como conhecido. Não é um painel com efeitos fixos de trecho nem identificação do efeito das concessões.

**Sensibilidades:** excluir 2025 e excluir segmentos não simuláveis. Uma conclusão que muda nessas escolhas deve ser descrita como sensível.

In [7]:
executar_etapa('analisar_exposicao.py')

"""PRF–PNCT: integração anual e modelos de contagem com exposição modelada.
Esta análise não identifica efeitos causais nem sinistros não registrados.
"""
from pathlib import Path
import json,calendar,math,datetime
import pandas as pd,numpy as np
from scipy.special import gammaln,digamma
from scipy.optimize import minimize
from scipy.stats import t as student_t
R=Path(__file__).resolve().parents[1];O=R/'dist'/'data';D=pd.read_pickle(R/'data'/'prf-limpa.pkl')
def write(n,obj):
 t=json.dumps(obj,ensure_ascii=False,separators=(',',':'),allow_nan=False,default=lambda x:x.item() if hasattr(x,'item') else str(x));(O/(n+'.json')).write_text(t,encoding='utf8');(O/(n+'.js')).write_text('window.'+n.upper()+'='+t+';',encoding='utf8')
panels=[];coverage=[]
for y in range(2022,2026):
 S=pd.read_csv(R/'data'/f'pnct-{y}-sheet2.csv');S.columns=S.columns.str.strip()
 for a,b in [('km_i_estim','vl_km_i_estim'),('km_f_estim','vl_km_f_estim'),('ext_estim','vl_ext_estim'),('classifica','classificação')]:
 

In [8]:
ex = resultado('exposicao')
print('COBERTURA')
print(pd.DataFrame(ex['coverage']).to_string(index=False))
print('TAXAS BRUTAS')
print(pd.DataFrame(ex['groups']).to_string(index=False))
for outcome, model in ex['models'].items():
    assert model['meta']['converged'], model['meta']
    print(outcome, model['meta'])
    print(pd.DataFrame(model['terms']).query("term == 'concessao'").to_string(index=False))
for name, model in ex['sensitivity'].items():
    print(name, pd.DataFrame(model['terms']).query("term == 'concessao'").to_string(index=False))
print('INTERPRETAÇÃO: a associação ajustada pode inverter a comparação bruta. A associação para mortes perde evidência estatística na sensibilidade sem segmentos não simuláveis; não permite concluir que concessão melhora ou piora causalmente a segurança.')


COBERTURA
 ano  linhas_originais  segmentos_elegiveis  ocorrencias_prf  ocorrencias_vinculadas  ambiguous  mortos_vinculados  segmentos_sem_registro    exposicao  vmda_min  vmda_max versao_snv
2022              6345                 3989            64606                   58406          0               4973                     859 1.273340e+11     406.0  148646.0  [202301B]
2023              6423                 4003            67766                   61407          0               5167                     825 1.342577e+11     422.0  152281.0  [202401A]
2024              8222                 5027            73202                   62518          0               5433                    1229 1.368628e+11     398.0   98677.0  [202401A]
2025              8222                 5030            72529                   61917          0               5384                    1258 1.390003e+11     436.0  111646.0  [202401A]
TAXAS BRUTAS
 ano                                  admin  segmentos  sinist

## 6. Validação temporal e verificações
O modelo logístico treina em 2022–2023 e testa 2024–2025, sem efeitos de ano. AUC mede discriminação; Brier mede erro probabilístico e deve ser comparado à referência calculada no treino. O modelo de contagens também usa separação temporal e reporta MAE e desvio Poisson.

Isso avalia previsão de **registros observados**, não detecção de eventos ocultos. Não usar como decisão operacional sem validação externa.

In [9]:
print('Validação NB2:', ex['validation'])
print('Modelo logístico:', resultado('modelo').keys())
# Verificações independentes de totais e denominadores.
panel = pd.read_csv(ROOT/'dist/exports/painel-prf-pnct.csv')
print('Colunas do painel:', list(panel.columns))
assert all(c['ocorrencias_vinculadas']<=c['ocorrencias_prf'] for c in ex['coverage'])
assert all(g['exposicao']>0 for g in ex['groups'])
assert sum(g['mortos'] for g in ex['groups'])==sum(c['mortos_vinculados'] for c in ex['coverage'])
for group in ex['groups']:
    assert np.isclose(group['taxa_mortes'], 1e8*group['mortos']/group['exposicao'])
executar_etapa('publicar_resultados.py')
print('Verificações concluídas.')


Validação NB2: {'train_years': [2022, 2023], 'test_years': [2024, 2025], 'n_train': 7857, 'n_test': 9859, 'observed': 10671.0, 'predicted': 10810.058509909943, 'mae': 0.9781186818571268, 'mean_poisson_deviance': 1.5989497973141493}
Modelo logístico: dict_keys(['meta', 'terms', 'validation', 'sensitivity', 'scope', 'exclusions'])
Colunas do painel: ['ano', 'ID', 'vl_codigo', 'uf', 'br', 'vl_km_i_estim', 'vl_km_f_estim', 'ext', 'vmda', 'exposicao', 'admin', 'ds_tipo_ad', 'ds_sup_fed', 'versao_snv', 'classificação', 'referencial', 'nao_simulavel', 'pesados', 'n', 'mortos', 'fatais', 'graves', 'cluster', 'concessao', 'dupla', 'log_vmd', 'pesados_10pp']
"""Figuras vetoriais, relatório, manifesto e verificações de consistência."""
from pathlib import Path
import json,hashlib,datetime,html,zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
R=Path(__file__).resolve().parents[1];OU

## 7. Governança: dados publicados e hipóteses ainda não testadas
O catálogo público abaixo informa instrumentos e investimentos. Sua presença no pacote não equivale a integração histórica ou índice validado de governança. Não combinar diferenças de cadastro de 2026 com resultados passados como se fossem exposição institucional contemporânea.

**Moderação proposta:** verificar se a associação de um arranjo/obrigação com taxas varia conforme capacidade operacional anterior. Termo `G×C`; apresentar taxas previstas e diferenças de efeitos com IC, porque a interação depende da escala. Requer histórico validado e suporte comum. Não estimada aqui.

**Mediação proposta:** obrigação/coordenação em t−1 → resposta em t → resultado em t+1. Requer ordem temporal, medida confiável do mediador e pressupostos sobre confundimento. Produto de coeficientes e ajuste de um SEM não bastam para identificar efeito indireto causal. Não estimada aqui.

**Contribuição a demonstrar:** conectar produção da informação e capacidade de resposta a mecanismos institucionais observáveis, confrontando explicações concorrentes. Complexidade estatística não substitui identificação.

Fontes públicas e protocolo: `dist/exports/protocolo-governanca.html`. Registro explícito das extensões: `dist/exports/registro-extensoes.json`.

In [10]:
catalogo = pd.read_csv(ROOT/'dist/exports/catalogo-governanca.csv')
print(catalogo.to_string(index=False))
print(json.dumps(json.loads((ROOT/'dist/exports/registro-extensoes.json').read_text(encoding='utf-8')), ensure_ascii=False, indent=2))


                                 arquivo  linhas_validas  linhas_malformadas                                                                                                                                                                                url_oficial                                                           sha256                                                                 uso_permitido_na_analise                                                                           limite
   antt-encerramento-investimentos-0.csv             253                   0                               https://dados.antt.gov.br/dataset/09020405-776b-4bbc-946e-410a37ab2a70/resource/37527602-5b8b-4b2d-aef7-f7ed59a2f473/download/encerramento-investimentos.csv 17e2739b5444ad7ea2534421c9ad0510c867a577f9ebfb7414cb716350959a51 Inventário de instrumentos, investimentos e cobertura cadastral; não índice de qualidade Não atribuir cadastro atual a períodos passados; conferir vigência e localização
  

## Referências e interpretação responsável
- PRF: https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf
- PNCT/DNIT: https://servicos.dnit.gov.br/dadospnct
- SNV/DNIT: https://www.gov.br/dnit/pt-br/assuntos/atlas-e-mapas/pnv-e-snv
- ANTT: https://dados.antt.gov.br/
- Alves et al. (2021), concessões e segurança: https://www.urbandemographics.org/files/2021_Alves_highway-concessions-road-safety.pdf
- Janstrup et al. (2016), vinculação e sub-registro: https://doi.org/10.1080/15389588.2015.1128533
- Bromley e Powell (2012), desacoplamento: https://www.patriciabromley.com/wp-content/uploads/2018/06/BromleyPowellDecoupling.pdf

Os achados deste notebook são exploratórios, sujeitos a confundimento, erro de exposição e seleção dos registros. Não constituem avaliação causal da autora, dos gestores ou das concessionárias. A contribuição e a publicação acadêmica exigem revisão teórica, validação externa e um desenho institucional executado.